In [1]:
import ast
import re
import pandas as pd
from datetime import datetime
from currency_converter import CurrencyConverter

In [2]:
INPUT_CSV  = "../data/bronze/Steam/steam_app_data.csv"
OUTPUT_CSV = "steam_clean5.csv"
YEAR_MIN = 2015
YEAR_MAX = 2025
YEAR_RE  = re.compile(r"\b(\d{4})\b")

In [3]:
def parse_date(s):
    s = (s or "").strip()
    for fmt in ("%d %b, %Y", "%b %d, %Y", "%d %B, %Y", "%B %d, %Y"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except:
            pass
    return None

def extract_date_string(cell):
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return None

    s = str(cell).strip()
    # Try dict-like
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, dict):
            if obj.get("coming_soon") is True:
                return None
            raw = (obj.get("date") or "").strip()
            date_str = parse_date(raw)
            return date_str or None
    except Exception as e:
        print(e)
        pass  # not a dict; fall through

    return s or None

def year_in_range_from_string(date_str):
    """
    Get any 4-digit year from the string and filter by YEAR_MIN..YEAR_MAX.
    If no year found, return False.
    """
    if not date_str:
        return False
    m = YEAR_RE.search(date_str)
    if not m:
        return False
    try:
        y = int(m.group(1))
        return YEAR_MIN <= y <= YEAR_MAX
    except Exception:
        return False

def dict_to_text(cell):
    """
    "[{'id': '1', 'description': 'Action'}, ...]" -> "Action|Adventure|..."
    """
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return ""
    try:
        val = ast.literal_eval(str(cell))
        if isinstance(val, list):
            parts = []
            for item in val:
                if isinstance(item, dict):
                    desc = item.get("description")
                    if desc:
                        parts.append(str(desc).strip())
            return "|".join(parts)
    except Exception:
        pass
    return ""

def coerce_int_or_empty(x):
    try:
        if x is None:
            return ""
        if isinstance(x, (int, float)) and not pd.isna(x):
            return int(x)
        xs = str(x).strip()
        return int(xs)
    except Exception:
        return ""

def parse_price_overview(cell):
    """
    "{'currency': 'USD', 'final_formatted': '$9.99', 'discount_percent': 0, ...}"
    -> (currency_code, discount_percent, final_price)
    """
    currency_code, discount_percent, final_price = "", "", ""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return currency_code, discount_percent, final_price
    try:
        dct = ast.literal_eval(str(cell))
        if isinstance(dct, dict):
            currency_code = str(dct.get("currency", "")).strip()
            discount_percent = coerce_int_or_empty(dct.get("discount_percent"))
            raw_price = str(dct.get("final_formatted", "")).strip()
            final_price = re.sub(r"[^0-9.]", "", raw_price)

    except Exception:
        pass
    return currency_code, discount_percent, final_price



In [4]:
def safe_convert(r):
        try:
            cur = r["currency_code"]
            price = r["final_price"]
            if price is not None and cur and cur != "USD":
                res = currency_converter.convert(price, cur, "USD")
                return round(res, 2)
            return price
        except Exception as e:
            # print(e)
            return None
        

currency_converter = CurrencyConverter()

In [21]:
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

# effective_date: extract string only (no conversion)
df["effective_date"] = df.get("release_date", "").apply(extract_date_string)


invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unknown>, line 0)
invalid syntax (<unk

In [22]:
out_of_range = df[df["effective_date"].apply(lambda x: not year_in_range_from_string(x))].copy()

In [23]:
# genres -> description-only joined with "|"
if "genres" in out_of_range.columns:
        out_of_range["genres"] = out_of_range["genres"].apply(dict_to_text)

if "categories" in out_of_range.columns:
        out_of_range["categories"] = out_of_range["categories"].apply(dict_to_text)


# price_overview -> currency_code, discount_percent, final_price
if "price_overview" in out_of_range.columns:
        cc, dp, fp = zip(*out_of_range["price_overview"].apply(parse_price_overview))
        out_of_range["currency_code"] = list(cc)
        out_of_range["discount_percent"] = list(dp)
        out_of_range["final_price"] = list(fp)
        out_of_range.drop(columns=["price_overview"], inplace=True)
        out_of_range["final_price_usd"] = out_of_range.apply(safe_convert, axis=1)

In [11]:
print(dict(out_of_range.dtypes.apply(lambda x: str(x))))

{'type': 'object', 'name': 'object', 'steam_appid': 'object', 'required_age': 'object', 'is_free': 'object', 'controller_support': 'object', 'dlc': 'object', 'detailed_description': 'object', 'about_the_game': 'object', 'short_description': 'object', 'fullgame': 'object', 'supported_languages': 'object', 'header_image': 'object', 'website': 'object', 'pc_requirements': 'object', 'mac_requirements': 'object', 'linux_requirements': 'object', 'legal_notice': 'object', 'drm_notice': 'object', 'ext_user_account_notice': 'object', 'developers': 'object', 'publishers': 'object', 'demos': 'object', 'packages': 'object', 'package_groups': 'object', 'platforms': 'object', 'metacritic': 'object', 'reviews': 'object', 'categories': 'object', 'genres': 'object', 'screenshots': 'object', 'movies': 'object', 'recommendations': 'object', 'achievements': 'object', 'release_date': 'object', 'support_info': 'object', 'background': 'object', 'content_descriptors': 'object', 'effective_date': 'object', 'cu

In [24]:
out_of_range = out_of_range[["name", "is_free", "developers", "publishers", "categories", "genres", "effective_date", "currency_code", "discount_percent", "final_price", "final_price_usd"]]

# Write out
out_of_range.to_csv('out_of_range_steam.csv', index=False)
print(f"Done. Saved {len(df)} rows to out_of_range_steam.csv")

Done. Saved 86540 rows to out_of_range_steam.csv


In [19]:
# Filter by year 2015..2025 based on the year number present in the string
df = df[df["effective_date"].apply(year_in_range_from_string)].copy()

# genres -> description-only joined with "|"
if "genres" in df.columns:
        df["genres"] = df["genres"].apply(dict_to_text)

if "categories" in df.columns:
        df["categories"] = df["categories"].apply(dict_to_text)

# price_overview -> currency_code, discount_percent, final_price
if "price_overview" in df.columns:
        cc, dp, fp = zip(*df["price_overview"].apply(parse_price_overview))
        df["currency_code"] = list(cc)
        df["discount_percent"] = list(dp)
        df["final_price"] = list(fp)
        df.drop(columns=["price_overview"], inplace=True)
        df["final_price_usd"] = df.apply(safe_convert, axis=1)

In [20]:
df = df[["name", "is_free", "developers", "publishers", "categories", "genres", "effective_date", "currency_code", "discount_percent", "final_price", "final_price_usd"]]
# Write out
df.to_csv(OUTPUT_CSV, index=False)
print(f"Done. Saved {len(df)} rows to {OUTPUT_CSV}")

Done. Saved 74594 rows to steam_clean5.csv
